# FighterAPI: a season of mma in five minutes

This notebook loads the free FighterAPI season pack (the last completed season, as CSV), answers a few questions offline, then asks the live API the same kind of question with a free key.

Data: FighterAPI (fighterapi.com), under the [FighterAPI Sample Data Licence](https://github.com/sports-data-hq/fighterapi-mcp/blob/main/DATA-LICENSE.md).

In [ ]:
import io, zipfile, urllib.request
import pandas as pd

URL = 'https://github.com/sports-data-hq/fighterapi-mcp/releases/download/season-pack/fighterapi-season-pack.zip'
z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(URL).read()))
tables = {n[:-4]: pd.read_csv(z.open(n)) for n in z.namelist() if n.endswith('.csv')}

def t(league, kind):
    """One table, e.g. t('ufc', 'players')."""
    return next(df for name, df in tables.items() if f'-{league}-' in name and name.endswith('-' + kind))

sorted(tables)[:12]

## Top of our Elo at season end

In [ ]:
t('ufc', 'ratings').head(15)

## Best in each weight class by our Elo

In [ ]:
r = t('ufc', 'ratings')
r.sort_values('elo', ascending=False).groupby('weight_class').head(1)[['weight_class', 'fighter', 'elo']]

## Finishers: share of wins by knockout or submission, minimum 3 wins

In [ ]:
f = t('ufc', 'fighters')
f[f.wins >= 3].assign(finish_rate=lambda d: (d.wins_ko_tko + d.wins_submission) / d.wins).sort_values('finish_rate', ascending=False)[['fighter', 'weight_class', 'wins', 'wins_ko_tko', 'wins_submission', 'finish_rate']].head(10)

## The same, live

Every earlier season, play-by-play, lineups, odds and today's games are on the FighterAPI API. Get a free key and paste it below: one key for every sport at [app.sportsdatahq.com](https://app.sportsdatahq.com/signup?utm_source=colab&utm_medium=notebook&utm_campaign=fighterapi-notebook) (1,000 credits a month), which this notebook uses.

In [ ]:
import getpass, json, urllib.request
KEY = getpass.getpass('API key: ')
req = urllib.request.Request('https://api.sportsdatahq.com/v1/mma/get_ratings', data=b'{}',
    headers={'x-api-key': KEY, 'content-type': 'application/json'})
res = urllib.request.urlopen(req)
print('credits used:', res.headers.get('x-credits-used'), '| left:', res.headers.get('x-credits-remaining'))
json.loads(res.read())['data']

Ask the same questions in plain English: add the MCP server `https://mcp.sportsdatahq.com/mcp` to Claude or ChatGPT, or install the mma skills from https://github.com/sports-data-hq/fighterapi-mcp.